In [ ]:
# CELL 1: Setup
import sys
sys.path.append('..')

import os
import random
import torch
import numpy as np
import matplotlib.pyplot as plt

from configs.config import Config
from data.splits import get_loaders, get_datasets
from models.bu_net import BUNet
from models.prototypical_segmentation import PrototypicalSegmentation
from configs.results_utils import load_results, print_comparison_table
from configs.model_utils import load_model_weights

Config.create_dirs()
print(f"✓ Device: {Config.DEVICE}")

In [ ]:
# CELL 2: Load All Results
baseline_results = load_results(Config.RESULTS_DIR, 'baseline_kshot_results.json')
proto_results = load_results(Config.RESULTS_DIR, 'prototypical_kshot_results.json')
maml_results = load_results(Config.RESULTS_DIR, 'maml_kshot_results.json')

for name, r in [('Baseline', baseline_results),
                ('Prototypical', proto_results),
                ('MAML', maml_results)]:
    if r:
        print(f"✓ {name} results loaded")
    else:
        print(f"✗ {name} results missing")

In [ ]:
# CELL 3: Method Comparison Chart
k_values = Config.K_SHOT_VALUES
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'Baseline Fine-tuning': '#2196F3',
          'Prototypical Networks': '#4CAF50',
          'MAML': '#FF9800'}

for name, results, color in [
    ('Baseline Fine-tuning', baseline_results, colors['Baseline Fine-tuning']),
    ('Prototypical Networks', proto_results, colors['Prototypical Networks']),
    ('MAML', maml_results, colors['MAML']),
]:
    if results is None:
        continue
    means = [results[str(k)]['mean'] for k in k_values]
    stds = [results[str(k)]['std'] for k in k_values]
    ax.errorbar(k_values, means, yerr=stds, marker='o', capsize=5,
                linewidth=2, markersize=8, color=color, label=name)

ax.set_xlabel('k (number of support examples)', fontsize=13)
ax.set_ylabel('Mean Tumor Dice Score', fontsize=13)
ax.set_title('Few-Shot Brain Tumor Segmentation: Method Comparison', fontsize=14)
ax.set_xticks(k_values)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(Config.RESULTS_DIR, 'method_comparison.png'), dpi=150)
plt.show()

In [ ]:
# CELL 4: Load Models for Visualization
train_loader, val_loader, test_loader = get_loaders(Config)
train_dataset, val_dataset, test_dataset = get_datasets(Config)

# Baseline model
baseline_model = BUNet(
    encoder_name=Config.ENCODER_NAME,
    in_channels=Config.IN_CHANNELS,
    num_classes=Config.NUM_CLASSES,
).to(Config.DEVICE)
load_model_weights(baseline_model, Config.CHECKPOINT_DIR, 'best_model.pth', Config.DEVICE)
baseline_model.eval()

# Prototypical model
proto_model = PrototypicalSegmentation(
    encoder_name=Config.ENCODER_NAME,
    in_channels=Config.IN_CHANNELS,
    num_classes=Config.NUM_CLASSES,
).to(Config.DEVICE)
load_model_weights(proto_model, Config.CHECKPOINT_DIR, 'prototypical_ep1000.pth', Config.DEVICE)
proto_model.eval()

print("✓ Models loaded for visualization")

In [ ]:
# CELL 5: Prediction Comparison
def show_predictions(models, model_names, dataset, sample_indices,
                     save_name='prediction_comparison.png'):
    """Side-by-side predictions from multiple models."""
    n_samples = len(sample_indices)
    n_cols = 3 + len(models)  # FLAIR, T1CE, GT, then one per model
    fig, axes = plt.subplots(n_samples, n_cols, figsize=(4 * n_cols, 4 * n_samples))

    if n_samples == 1:
        axes = axes[np.newaxis, :]

    for row, idx in enumerate(sample_indices):
        sample = dataset[idx]
        image = sample['image']
        mask = sample['mask']

        # Input channels
        axes[row, 0].imshow(image[0], cmap='gray')
        axes[row, 0].set_title('FLAIR' if row == 0 else '')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(image[1], cmap='gray')
        axes[row, 1].set_title('T1CE' if row == 0 else '')
        axes[row, 1].axis('off')

        # Ground truth
        axes[row, 2].imshow(mask, cmap='viridis', vmin=0, vmax=3)
        axes[row, 2].set_title('Ground Truth' if row == 0 else '')
        axes[row, 2].axis('off')

        # Model predictions
        for col, (model, name) in enumerate(zip(models, model_names)):
            with torch.no_grad():
                inp = image.unsqueeze(0).to(Config.DEVICE)
                pred = torch.argmax(model(inp), dim=1).squeeze().cpu()
            axes[row, 3 + col].imshow(pred, cmap='viridis', vmin=0, vmax=3)
            axes[row, 3 + col].set_title(name if row == 0 else '')
            axes[row, 3 + col].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(Config.RESULTS_DIR, save_name), dpi=150)
    plt.show()

# Pick slices that contain tumor
tumor_indices = []
for i in range(0, len(val_dataset), 50):
    sample = val_dataset[i]
    if sample['mask'].max() > 0:
        tumor_indices.append(i)
    if len(tumor_indices) >= 4:
        break

show_predictions(
    models=[baseline_model, proto_model],
    model_names=['Baseline', 'Prototypical'],
    dataset=val_dataset,
    sample_indices=tumor_indices,
)

In [ ]:
# CELL 6: Prototype Visualization
def visualize_prototypes(model, dataset, k=5, save_name='prototype_viz.png'):
    """Show what the prototypical network learns as class prototypes."""
    # Sample support set
    indices = random.sample(range(len(dataset)), k)
    support_imgs = torch.stack([dataset[i]['image'] for i in indices]).to(Config.DEVICE)
    support_masks = torch.stack([dataset[i]['mask'] for i in indices]).to(Config.DEVICE)

    prototypes = model.compute_prototypes(support_imgs, support_masks)

    class_names = ['Background', 'NCR/NET', 'Edema', 'Enhancing']
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    for i, (proto, name) in enumerate(zip(prototypes, class_names)):
        # Show prototype as a 1D feature vector heatmap
        proto_np = proto.cpu().numpy()
        axes[i].bar(range(len(proto_np)), proto_np, color=plt.cm.viridis(i / 3))
        axes[i].set_title(f'{name} Prototype')
        axes[i].set_xlabel('Feature Dimension')
        axes[i].set_ylabel('Value')
        axes[i].tick_params(axis='x', labelsize=6)

    plt.suptitle(f'Learned Class Prototypes (k={k} support examples)', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(Config.RESULTS_DIR, save_name), dpi=150)
    plt.show()

visualize_prototypes(proto_model, val_dataset, k=5)

In [ ]:
# CELL 7: Error Analysis
def error_analysis(model, dataset, num_samples=100, save_name='error_analysis.png'):
    """Analyze where the model fails — per-class Dice distribution."""
    from configs.metrics import dice_score

    all_dice = {c: [] for c in range(4)}

    indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))
    for idx in indices:
        sample = dataset[idx]
        image = sample['image'].unsqueeze(0).to(Config.DEVICE)
        mask = sample['mask'].unsqueeze(0).to(Config.DEVICE)

        with torch.no_grad():
            pred = torch.argmax(model(image), dim=1)

        scores = dice_score(pred, mask)
        for c in range(4):
            all_dice[c].append(scores[f'dice_class_{c}'])

    # Box plot
    class_names = ['Background', 'NCR/NET', 'Edema', 'Enhancing']
    fig, ax = plt.subplots(figsize=(8, 5))
    data = [all_dice[c] for c in range(4)]
    bp = ax.boxplot(data, labels=class_names, patch_artist=True)

    colors_box = ['#90CAF9', '#EF9A9A', '#A5D6A7', '#FFE082']
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)

    ax.set_ylabel('Dice Score', fontsize=12)
    ax.set_title('Per-Class Dice Score Distribution (Baseline)', fontsize=14)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(Config.RESULTS_DIR, save_name), dpi=150)
    plt.show()

error_analysis(baseline_model, val_dataset)

In [ ]:
# CELL 8: Summary Table
print_comparison_table([
    ('Baseline Fine-tuning', baseline_results),
    ('Prototypical Networks', proto_results),
    ('MAML', maml_results),
], Config.K_SHOT_VALUES)